# StayNest - Session 7 Assignment (Delta Lake & Lakehouse)
Work through the 8 tasks in order. Read the Assignment Questions PDF for the full
detail and acceptance criteria. Fill each `# TODO` cell, run it, and keep the output
visible. Runs on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `bookings_updates.csv` to a Volume, set `BASE`,
`CATALOG`, `SCHEMA`, and run this cell. Expect 12000 / 200 / 200.

In [0]:
BASE    = "/Volumes/cb_catalog/default/staynest_v2_vol"
CATALOG = "cb_catalog"
SCHEMA  = "default"
FQN = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

read_csv = lambda name: (spark.read
    .option("header", True).option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df = read_csv("bookings")
hotels_df   = read_csv("hotels")
updates_df  = read_csv("bookings_updates")

print(f"bookings: {bookings_df.count()}, hotels: {hotels_df.count()}, "
      f"updates: {updates_df.count()}")

bookings: 12000, hotels: 200, updates: 200


## Task 1 - Read the plan and force a broadcast join
Join bookings to hotels and call `.explain()` to see the plan. Then force a
broadcast join with `broadcast(hotels_df)` and `.explain()` again. In a comment,
say which join each plan used and why broadcast avoids a shuffle.
(Tip: hotels also has a `city` column, so `hotels_df.drop("city")` before joining.)

In [0]:
from pyspark.sql.functions import broadcast

# Remove duplicate city column from hotels
hotels = hotels_df.drop("city")

# -----------------------------
# 1. Regular Join
# -----------------------------
regular_join = bookings_df.join(
    hotels,
    on="hotel_id",
    how="inner"
)

print("===== Regular Join Execution Plan =====")
regular_join.explain(True)


# -----------------------------
# 2. Broadcast Join
# -----------------------------
broadcast_join = bookings_df.join(
    broadcast(hotels),
    on="hotel_id",
    how="inner"
)

print("\n===== Broadcast Join Execution Plan =====")
broadcast_join.explain(True)

# Regular Join:
# Catalyst typically chooses a SortMergeJoin (or ShuffledHashJoin),which requires shuffling both DataFrames so matching hotel_id values
# are located in the same partition.

# Broadcast Join:
# Catalyst uses BroadcastHashJoin because the hotels table is small.Spark broadcasts the entire hotels DataFrame to every executor, so
# each executor can perform the join locally without shuffling the # much larger bookings DataFrame. This reduces network traffic and
# improves performance.

===== Regular Join Execution Plan =====
== Parsed Logical Plan ==
'Join UsingJoin(Inner, [hotel_id])
:- Relation [booking_id#11200,customer_id#11201,hotel_id#11202,booking_date#11203,city#11204,nights#11205,amount#11206,status#11207] csv
+- Project [hotel_id#11231, hotel_name#11232, category#11234, star_rating#11235]
   +- Relation [hotel_id#11231,hotel_name#11232,city#11233,category#11234,star_rating#11235] csv

== Analyzed Logical Plan ==
hotel_id: int, booking_id: int, customer_id: int, booking_date: date, city: string, nights: int, amount: double, status: string, hotel_name: string, category: string, star_rating: double
Project [hotel_id#11202, booking_id#11200, customer_id#11201, booking_date#11203, city#11204, nights#11205, amount#11206, status#11207, hotel_name#11232, category#11234, star_rating#11235]
+- Join Inner, (hotel_id#11202 = hotel_id#11231)
   :- Relation [booking_id#11200,customer_id#11201,hotel_id#11202,booking_date#11203,city#11204,nights#11205,amount#11206,status#1

## Task 2 - Create a Delta table, then read its history
Write `bookings_df` as a managed Delta table with `saveAsTable`. Then create some
history: run an `UPDATE` (set pending to completed) and a `DELETE` (remove
cancelled). Show `DESCRIBE HISTORY` and point out the versioned commits.

In [0]:

from delta.tables import DeltaTable

# ------------------------------------
# 1. Save bookings_df as a managed Delta table
# ------------------------------------
bookings_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bookings_delta")

print("Managed Delta table created and sample data below.")

# View the data
spark.sql("SELECT * FROM bookings_delta LIMIT 5").show()

#Update all pending bookings to completed.
spark.sql("""
UPDATE bookings_delta
SET status = 'completed'
WHERE status = 'pending'
""")


#Remove cancelled bookings.
spark.sql("""
DELETE FROM bookings_delta
WHERE status = 'cancelled'
""")

print("Delta History changes post status Update and pending record deletes")
#Display Delta history  
history_df = spark.sql("""
DESCRIBE HISTORY bookings_delta
""")

display(history_df)

Managed Delta table created and sample data below.
+----------+-----------+--------+------------+---------+------+--------+---------+
|booking_id|customer_id|hotel_id|booking_date|     city|nights|  amount|   status|
+----------+-----------+--------+------------+---------+------+--------+---------+
|   9000000|     701600|    3095|  2025-11-27|   Jaipur|     4| 6087.65|completed|
|   9000001|     700065|    3057|  2025-11-06|    Delhi|     1| 8211.19|cancelled|
|   9000002|     701392|    3187|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|
|   9000003|     700867|    3112|  2025-03-22|Bengaluru|     5| 7880.62|completed|
|   9000004|     701521|    3043|  2025-04-19|   Mumbai|     5|21021.51|  pending|
+----------+-----------+--------+------------+---------+------+--------+---------+

Delta History changes post status Update and pending record deletes


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2026-08-01T00:09:15.000Z,74629125555835,sandeep.antony007@gmail.com,DELETE,"Map(predicate -> [""(status#13834 = cancelled)""])",null,List(2866543972767295),50235404-ede0-45a4-8c83-0c2049dcd642,0801-000326-xnznz8lb-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1288, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 892, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 396)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-08-01T00:09:12.000Z,74629125555835,sandeep.antony007@gmail.com,UPDATE,"Map(predicate -> [""(status#13410 = pending)""])",null,List(2866543972767295),01b0ec9f-0c4b-4465-bba7-74bb6b46ef41,0801-000326-xnznz8lb-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2345, numDeletionVectorsUpdated -> 0, scanTimeMs -> 911, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 1433)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-01T00:09:08.000Z,74629125555835,sandeep.antony007@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2866543972767295),bd782262-9c95-44ad-ad1b-667bf6e009b0,0801-000326-xnznz8lb-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 101467, numDeletionVectorsRemoved -> 0, numOutputRows -> 12000, numOutputBytes -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-01T00:07:53.000Z,74629125555835,sandeep.antony007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2866543972767295),850584d0-f135-4091-81f4-d154473b2e40,0801-000326-xnznz8lb-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 125764, p25FileSize -> 101467, numDeletionVectorsRemoved -> 1, minFileSize -> 101467, numAddedFiles -> 1, maxFileSize -> 101467, p75FileSize -> 101467, p50FileSize -> 101467, numAddedBytes -> 101467)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-01T00:07:50.000Z,74629125555835,sandeep.antony007@gmail.com,DELETE,"Map(predicate -> [""(status#12519 = cancelled)""])",null,List(2866543972767295),850584d0-f135-4091-81f4-d154473b2e40,0801-000326-xnznz8lb-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1706, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 1127, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 578)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-01T00:07:48.000Z,74629125555835,sandeep.antony007@gmail.com,UPDATE,"Map(predicate -> [""(status#12070 = pending)""])",null,List(2866543972767295),438edfb5-8a4e-4f4b-bb8d-548e45295405,0801-000326-xnznz8lb-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4796, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2720, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 2039)",null,Databricks-Runtime/18.x

## Task 3 - Time travel and RESTORE
Read the table as it was at **version 0** (before your UPDATE and DELETE) and show
its count. Then `RESTORE` the table to version 0 and confirm the count is back.
Show that RESTORE appears as a new commit in the history.

In [0]:
# Read the table as it existed at Version 0
version0_df = (
    spark.read
         .format("delta")
         .option("versionAsOf", 0)
         .table("bookings_delta")
)

version0_count = version0_df.count()
print(f"Row count at Version 0: {version0_count}")

# Reading the current row count
current_count = spark.table("bookings_delta").count()
print(f"Current row count: {current_count}")

#Restore the Table to Version 0
spark.sql("""
RESTORE TABLE bookings_delta TO VERSION AS OF 0
""")

#Confirm the Row Count is Back
restored_count = spark.table("bookings_delta").count()  
print(f"Row count after RESTORE: {restored_count}")

#Confirming the RESTORE History
history = spark.sql("""
DESCRIBE HISTORY bookings_delta
""")
display(history)

Row count at Version 0: 12000
Current row count: 10563
Row count after RESTORE: 12000


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-08-01T00:20:39.000Z,74629125555835,sandeep.antony007@gmail.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(2866543972767295),babc0dcd-08cc-45b4-87bf-95f048eea256,0801-000326-xnznz8lb-v2n,8,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 101467, numRemovedFiles -> 1, restoredFilesSize -> 113163, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
8,2026-08-01T00:09:16.000Z,74629125555835,sandeep.antony007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2866543972767295),50235404-ede0-45a4-8c83-0c2049dcd642,0801-000326-xnznz8lb-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 125764, p25FileSize -> 101467, numDeletionVectorsRemoved -> 1, minFileSize -> 101467, numAddedFiles -> 1, maxFileSize -> 101467, p75FileSize -> 101467, p50FileSize -> 101467, numAddedBytes -> 101467)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
7,2026-08-01T00:09:15.000Z,74629125555835,sandeep.antony007@gmail.com,DELETE,"Map(predicate -> [""(status#13834 = cancelled)""])",null,List(2866543972767295),50235404-ede0-45a4-8c83-0c2049dcd642,0801-000326-xnznz8lb-v2n,6,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1288, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1437, scanTimeMs -> 892, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 396)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
6,2026-08-01T00:09:12.000Z,74629125555835,sandeep.antony007@gmail.com,UPDATE,"Map(predicate -> [""(status#13410 = pending)""])",null,List(2866543972767295),01b0ec9f-0c4b-4465-bba7-74bb6b46ef41,0801-000326-xnznz8lb-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2345, numDeletionVectorsUpdated -> 0, scanTimeMs -> 911, numAddedFiles -> 1, numUpdatedRows -> 903, numAddedBytes -> 12601, rewriteTimeMs -> 1433)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
5,2026-08-01T00:09:08.000Z,74629125555835,sandeep.antony007@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2866543972767295),bd782262-9c95-44ad-ad1b-667bf6e009b0,0801-000326-xnznz8lb-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 101467, numDeletionVectorsRemoved -> 0, numOutputRows -> 12000, numOutputBytes -> 113163)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-01T00:07:53.000Z,74629125555835,sandeep.antony007@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2866543972767295),850584d0-f135-4091-81f4-d154473b2e40,0801-000326-xnznz8lb-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 125764, p25FileSize -> 101467, numDeletionVectorsRemoved -> 1, minFileSize -> 101467, numAddedFiles -> 1, maxFileSize -> 101467, p75FileSize -> 101467, p50FileSize -> 101467, numAddedBytes -> 101467)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-01T00:07:50.000Z,74629125555835,sandeep.antony007@gmail.com,DELETE,"Map(predicate -> [""(status#12519 = cancelled)

## Task 4 - OPTIMIZE and ZORDER
Run `OPTIMIZE` on your Delta table to compact files. Then run
`OPTIMIZE ... ZORDER BY (city)`. In a comment, say what OPTIMIZE does and why
`city` is a good ZORDER column but `status` would not be.

In [0]:
# ------------------------------------
# 1. Compact small files with OPTIMIZE
# ------------------------------------
spark.sql("""
OPTIMIZE bookings_delta
""")

print("OPTIMIZE completed successfully.")


# ------------------------------------
# 2. Optimize with Z-Ordering on city
# ------------------------------------
spark.sql("""
OPTIMIZE bookings_delta
ZORDER BY (city)
""")

# OPTIMIZE:
# OPTIMIZE compacts many small Delta files into fewer, larger files which reduces file-opening overhead and improves query performance.

# Why city?
# City is commonly used in filters and has enough distinct values to benefit from data clustering.

# Why not status?
# Status has very low cardinality (completed, pending, cancelled) and provides little improvement in data skipping or query performance.

print("OPTIMIZE with ZORDER completed successfully.")


OPTIMIZE completed successfully.
OPTIMIZE with ZORDER completed successfully.


## Task 5 - Bronze: land the raw data
Write the raw bookings to a `bronze_bookings` Delta table, keeping every row and
adding an `ingested_at` timestamp column.

In [0]:
from pyspark.sql.functions import current_timestamp

# ----------------------------------------
# Add ingestion timestamp to raw bookings
# ----------------------------------------
bronze_bookings_df = bookings_df.withColumn(
    "ingested_at",
    current_timestamp()
)

# ----------------------------------------
# Write as a managed Delta table
# ----------------------------------------
bronze_bookings_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_bookings")

print("Bronze Delta table created successfully")

# Display a few records
print("Below are some sample records")
spark.table("bronze_bookings").show(5, truncate=False)

# Verify the schema
print("----->Table Schema--->")
spark.table("bronze_bookings").printSchema()

# Verify the row count
print("Row count:", spark.table("bronze_bookings").count())


Bronze Delta table created successfully
Below are some sample records
+----------+-----------+--------+------------+---------+------+--------+---------+--------------------------+
|booking_id|customer_id|hotel_id|booking_date|city     |nights|amount  |status   |ingested_at               |
+----------+-----------+--------+------------+---------+------+--------+---------+--------------------------+
|9000000   |701600     |3095    |2025-11-27  |Jaipur   |4     |6087.65 |completed|2026-08-01 00:27:42.808027|
|9000001   |700065     |3057    |2025-11-06  |Delhi    |1     |8211.19 |cancelled|2026-08-01 00:27:42.808027|
|9000002   |701392     |3187    |2025-08-21  |Jaipur   |2     |7176.52 |cancelled|2026-08-01 00:27:42.808027|
|9000003   |700867     |3112    |2025-03-22  |Bengaluru|5     |7880.62 |completed|2026-08-01 00:27:42.808027|
|9000004   |701521     |3043    |2025-04-19  |Mumbai   |5     |21021.51|pending  |2026-08-01 00:27:42.808027|
+----------+-----------+--------+------------+----

## Task 6 - Silver: clean and conform
Build `silver_bookings` from bronze: keep only completed bookings and join the
hotel dimension to add `category`, `star_rating`, and the hotel name. Drop the
duplicate `city` from the hotel side so the join has a single `city`.

In [0]:
from pyspark.sql.functions import col

# ----------------------------------------
# Read Bronze and Hotel tables
# ----------------------------------------
bronze_bookings = spark.table("bronze_bookings")
hotels = hotels_df.drop("city")   # Remove duplicate city column

# ----------------------------------------
# Keep only completed bookings
# ----------------------------------------
completed_bookings = bronze_bookings.filter(
    col("status") == "completed"
)

# ----------------------------------------
# Join with hotel dimension
# ----------------------------------------
silver_bookings = (
    completed_bookings.join(
        hotels,
        on="hotel_id",
        how="inner"
    )
)

# ----------------------------------------
# Write Silver table as Delta
# ----------------------------------------
silver_bookings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_bookings")

print("Silver Delta table created successfully.")

# Display sample records
spark.table("silver_bookings").show(5, truncate=False)

# Verifying schema
spark.table("silver_bookings").printSchema()

# Row count check
print("Row count:", spark.table("silver_bookings").count())


Silver Delta table created successfully.
+--------+----------+-----------+------------+---------+------+--------+---------+--------------------------+----------------+--------+-----------+
|hotel_id|booking_id|customer_id|booking_date|city     |nights|amount  |status   |ingested_at               |hotel_name      |category|star_rating|
+--------+----------+-----------+------------+---------+------+--------+---------+--------------------------+----------------+--------+-----------+
|3095    |9000000   |701600     |2025-11-27  |Jaipur   |4     |6087.65 |completed|2026-08-01 00:27:42.808027|Orchid Suites   |Budget  |3.8        |
|3112    |9000003   |700867     |2025-03-22  |Bengaluru|5     |7880.62 |completed|2026-08-01 00:27:42.808027|Grand Stay      |Budget  |3.6        |
|3012    |9000006   |701336     |2025-11-25  |Delhi    |7     |70999.15|completed|2026-08-01 00:27:42.808027|Orchid Residency|Luxury  |4.1        |
|3127    |9000007   |700868     |2025-04-10  |Mumbai   |2     |15693.64

## Task 7 - Gold: business-ready aggregate
From silver, build a `gold_city_revenue` Delta table: bookings and total revenue
per city, ordered by revenue.

In [0]:
from pyspark.sql.functions import count, sum, round, col

# ----------------------------------------
# Read the Silver table
# ----------------------------------------
silver_df = spark.table("silver_bookings")

# ----------------------------------------
# Aggregate bookings and revenue by city
# ----------------------------------------
gold_city_revenue = (
    silver_df.groupBy("city")
    .agg(
        count("booking_id").alias("total_bookings"),
        round(sum("amount"), 2).alias("total_revenue")
    )
    .orderBy(col("total_revenue").desc())
)

# ----------------------------------------
# Write as a managed Delta table
# ----------------------------------------
gold_city_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_city_revenue")

print("Gold Delta table created successfully.")

# Verify the row count
print("Number of cities:", spark.table("gold_city_revenue").count())

# Display the results
spark.table("gold_city_revenue").show(truncate=False)


Gold Delta table created successfully.
Number of cities: 10
+---------+--------------+-------------+
|city     |total_bookings|total_revenue|
+---------+--------------+-------------+
|Goa      |2546          |4.459670179E7|
|Mumbai   |1715          |3.624122112E7|
|Delhi    |1174          |2.631428154E7|
|Jaipur   |979           |2.443685313E7|
|Bengaluru|1318          |2.267013697E7|
|Udaipur  |691           |1.209442742E7|
|Rishikesh|407           |8606121.58   |
|Manali   |480           |6235480.68   |
|Munnar   |244           |3979216.11   |
|Anantapur|106           |2257080.65   |
+---------+--------------+-------------+



## Task 8 - Incremental load with MERGE
You have today's batch in `updates_df` (150 changed bookings + 50 new ones).
`MERGE` it into your Delta table: update matched booking_ids, insert new ones, in
one command. Report the row count before and after (it should grow by the 50 new).

In [0]:
from delta.tables import DeltaTable

# ----------------------------------------
# Get row count before MERGE
# ----------------------------------------
before_count = spark.table("bookings_delta").count()
print(f"Row count before MERGE: {before_count}")

# ----------------------------------------
# Reference the Delta table
# ----------------------------------------
delta_table = DeltaTable.forName(spark, "bookings_delta")

# ----------------------------------------
# MERGE (Update existing + Insert new)
# ----------------------------------------
(
    delta_table.alias("target")
    .merge(
        updates_df.alias("source"),
        "target.booking_id = source.booking_id"
    )
    .whenMatchedUpdate(set={
        "customer_id": "source.customer_id",
        "hotel_id": "source.hotel_id",
        "booking_date": "source.booking_date",
        "city": "source.city",
        "nights": "source.nights",
        "amount": "source.amount",
        "status": "source.status"
    })
    .whenNotMatchedInsert(values={
        "booking_id": "source.booking_id",
        "customer_id": "source.customer_id",
        "hotel_id": "source.hotel_id",
        "booking_date": "source.booking_date",
        "city": "source.city",
        "nights": "source.nights",
        "amount": "source.amount",
        "status": "source.status"
    })
    .execute()
)

# ----------------------------------------
# Get row count after MERGE
# ----------------------------------------
after_count = spark.table("bookings_delta").count()

print(f"Row count after MERGE: {after_count}")
print(f"Rows added: {after_count - before_count}")


Row count before MERGE: 12000
Row count after MERGE: 12050
Rows added: 50
